# 05 Initialization and Tanh Saturation

Part 3: Başlangıç loss'unun yüksekliği, tanh doyumu (saturation), ölü gradyanlar ve Kaiming Normal init.


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

block_size = 3
X, Y = [], []
for w in words:
    context = [0] * block_size
    for ch in w + ".":
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

# 1. Olceksiz baslatmada yuksek loss ve tanh doyumu
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)

emb = C[X[:1000]].view(-1, 30)
hpreact = emb @ W1 + b1
h = torch.tanh(hpreact)
logits = h @ W2 + b2
loss_unscaled = F.cross_entropy(logits, Y[:1000])

print("Olceksiz baslangic loss:", loss_unscaled.item())
print("Beklenen baslangic loss (-log(1/27)):", -torch.log(torch.tensor(1/27)).item())
print("Doymus tanh orani (|h| > 0.99):", (h.abs() > 0.99).float().mean().item())

plt.figure(figsize=(12, 4))
plt.subplot(121)
plt.hist(hpreact.view(-1).tolist(), 50, density=True)
plt.title("Pre-aktivasyonlar (Olceksiz)")
plt.subplot(122)
plt.hist(h.view(-1).tolist(), 50, density=True)
plt.title("Tanh Aktivasyonlari (Uclarda Doymus)")
plt.show()

# 2. Kaiming Normal init ve W2 olcekleme ile cozum
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 10), generator=g)
# Tanh icin gain = 5/3, fan_in = 30
W1 = torch.randn((30, 200), generator=g) * (5/3) / (30**0.5)
b1 = torch.randn(200, generator=g) * 0.01
# W2 kucultulerek esit baslangic olasiliklari saglanir
W2 = torch.randn((200, 27), generator=g) * 0.01
b2 = torch.randn(27, generator=g) * 0

emb = C[X[:1000]].view(-1, 30)
hpreact = emb @ W1 + b1
h = torch.tanh(hpreact)
logits = h @ W2 + b2
loss_kaiming = F.cross_entropy(logits, Y[:1000])

print("Kaiming init sonrasi baslangic loss:", loss_kaiming.item())
print("Kaiming sonrasi doymus tanh orani:", (h.abs() > 0.99).float().mean().item())

plt.figure(figsize=(12, 4))
plt.subplot(121)
plt.hist(hpreact.view(-1).tolist(), 50, density=True)
plt.title("Pre-aktivasyonlar (Kaiming Init)")
plt.subplot(122)
plt.hist(h.view(-1).tolist(), 50, density=True)
plt.title("Tanh Aktivasyonlari (Duzenli Dagilim)")
plt.show()
